# Contract Intelligence Multi-Agent System
## Assignment Notebook - 8-Week Capstone Project

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Welcome to Your Capstone Assignment!

This notebook is your guided journey to building a **production-grade multi-agent contract intelligence system**. Unlike the master solution, YOU will implement the core functionality.

### How This Assignment Works

1. **Scaffolding Provided**: Class structures, function signatures, and imports are given
2. **TODO Markers**: Look for `# TODO:` comments - these are YOUR tasks
3. **Hints**: Each section has hints to guide you (but not give away the answer)
4. **Validation Cells**: Run these to check if your implementation is correct
5. **Expected Output**: Sample outputs show what success looks like

### Difficulty Progression

| Week | Difficulty | Focus |
|------|-----------|-------|
| 1 | Easy | Environment setup (mostly provided) |
| 2 | Easy-Medium | Document processing methods |
| 3 | Medium | Vector store search functions |
| 4 | Medium-Hard | Implement agents from scratch |
| 5 | Hard | Multi-agent orchestration |
| 6 | Medium | Build knowledge graph |
| 7 | Medium | Add observability metrics |
| 8 | Hard | Integrate everything |

### Grading Rubric

- **Week 1-2**: 15% (Foundation)
- **Week 3-4**: 30% (Core Components)  
- **Week 5-6**: 30% (Advanced Features)
- **Week 7-8**: 25% (Production Polish)

Let's begin!

---

# WEEK 1: Environment & Foundations

**Difficulty: Easy** - Most code is provided. Focus on understanding.

---

## Learning Objectives

By the end of Week 1, you will:
- [ ] Set up all required packages
- [ ] Configure API keys securely
- [ ] Initialize Langfuse for observability
- [ ] Create traced wrapper functions
- [ ] Explore the contract data structure

---

## 1.1 Package Installation

This section is **provided** - just run the cell to install packages.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION (PROVIDED)
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("openpyxl", "openpyxl"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

## 1.2 Environment Configuration

This section is **provided** - handles environment detection and imports.

In [ ]:
# ============================================================================
# WEEK 1.2: ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Environment detection
IN_COLAB = 'google.colab' in sys.modules

print(f"Runtime Environment: {'Google Colab' if IN_COLAB else 'Local/Other'}")
print(f"Python Version: {sys.version.split()[0]}")

In [ ]:
# ============================================================================
# API KEY CONFIGURATION (PROVIDED)
# ============================================================================

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "contract_intelligence"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

PROJECT_NAME = "contract-intelligence-system"
DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

## 1.3 Langfuse Initialization

**YOUR FIRST TODO!** Initialize the Langfuse client.

### Hints:
- Import `Langfuse` from `langfuse`
- Import `observe` and `langfuse_context` from `langfuse.decorators`
- Use environment variables for credentials
- Call `auth_check()` to verify connection

In [ ]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================
# TODO: Import necessary Langfuse modules

from langfuse import Langfuse
from langfuse.decorators import observe, langfuse_context
from openai import OpenAI

# TODO: Initialize the Langfuse client
# Hint: Use os.environ.get() to retrieve keys
# The constructor takes: secret_key, public_key, host

langfuse = None  # TODO: Replace with actual initialization

# YOUR CODE HERE:
# langfuse = Langfuse(
#     secret_key=???,
#     public_key=???,
#     host=???
# )

# TODO: Verify connection
# Hint: Use langfuse.auth_check() in a try/except block

# YOUR CODE HERE:


# TODO: Create a unique session ID
# Hint: Use datetime.now().strftime() to create a unique identifier
# Format: "contract-intel-YYYYMMDD-HHMMSS"

SESSION_ID = None  # TODO: Replace with actual session ID

# YOUR CODE HERE:


print(f"Session ID: {SESSION_ID}")

### Expected Output for 1.3:
```
Langfuse connection verified!
Session ID: contract-intel-20250127-143052
```

## 1.4 Traced Wrapper Functions

**TODO:** Implement traced versions of OpenAI calls.

### Hints:
- Create a trace with `langfuse.trace()`
- Create a generation span with `trace.generation()`
- Use `openai_client.embeddings.create()` for embeddings
- End the generation with usage stats

In [ ]:
# ============================================================================
# WEEK 1.4: TRACED EMBEDDING FUNCTION
# ============================================================================

# Initialize OpenAI client (PROVIDED)
openai_client = OpenAI()

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Args:
        text: The text to embed
        trace_name: Identifier for this trace

    Returns:
        List of floats (embedding vector)

    TODO: Implement this function

    Steps:
    1. Create a trace with langfuse.trace()
    2. Create a generation span for the embedding call
    3. Call openai_client.embeddings.create()
    4. End the generation with output metadata
    5. Return the embedding
    """

    # TODO: Create trace
    # Hint: trace = langfuse.trace(name=..., session_id=SESSION_ID, metadata={...})

    # YOUR CODE HERE:
    trace = None


    # TODO: Create generation span
    # Hint: generation = trace.generation(name="openai-embedding", model="text-embedding-3-small", input=...)

    # YOUR CODE HERE:
    generation = None


    # TODO: Make API call
    # Hint: response = openai_client.embeddings.create(model="text-embedding-3-small", input=text)

    # YOUR CODE HERE:
    embedding = None


    # TODO: End generation with metadata
    # Hint: generation.end(output={...}, usage={...})

    # YOUR CODE HERE:


    return embedding

print("traced_embedding() function defined")

In [ ]:
# ============================================================================
# WEEK 1.4: TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(prompt: str, trace_name: str = "completion") -> str:
    """
    Generate completion with full Langfuse tracing.

    Args:
        prompt: The prompt to complete
        trace_name: Identifier for this trace

    Returns:
        Generated text

    TODO: Implement this function (similar pattern to traced_embedding)
    """

    # TODO: Create trace
    # YOUR CODE HERE:
    trace = None


    # TODO: Create generation span
    # YOUR CODE HERE:
    generation = None


    # TODO: Make API call using openai_client.chat.completions.create()
    # Hint: Use model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}]

    # YOUR CODE HERE:
    response_text = None


    # TODO: End generation with usage stats
    # YOUR CODE HERE:


    return response_text

print("traced_completion() function defined")

## 1.5 Contract Document Taxonomy

**PROVIDED** - Study this structure carefully, you'll need it later.

In [ ]:
# ============================================================================
# WEEK 1.5: CONTRACT TAXONOMY (PROVIDED)
# ============================================================================

CONTRACT_CATEGORIES = {
    'master_agreements': {
        'path': 'Master level agreements',
        'description': 'Foundation contracts establishing overall relationship',
        'document_types': ['MSA', 'NDA', 'Rate Cards'],
        'risk_focus': ['legal', 'compliance'],
    },
    'transaction_contracts': {
        'path': 'Transaction level contract',
        'description': 'Project-specific agreements under master agreements',
        'document_types': ['SOW', 'Work Orders', 'Renewals', 'Amendments'],
        'risk_focus': ['operational', 'financial'],
    },
    'commercial_docs': {
        'path': 'Commercial docs',
        'description': 'Pricing and commercial terms documentation',
        'document_types': ['Pricing Tables', 'Rate Schedules'],
        'risk_focus': ['financial'],
    },
    'financial_billing': {
        'path': 'Financial and Billing docs',
        'description': 'Invoices, payment records, and billing schedules',
        'document_types': ['Invoices', 'Credit Notes', 'Payment Schedules'],
        'risk_focus': ['financial'],
    },
    'operational_docs': {
        'path': 'Operational service delivery docs',
        'description': 'Service delivery and operational documentation',
        'document_types': ['Service Reports', 'SLA Reports'],
        'risk_focus': ['operational'],
    },
    'compliance_docs': {
        'path': 'Compliance and policy documents',
        'description': 'Security, compliance, and policy documentation',
        'document_types': ['InfoSec Controls', 'Vendor Policies'],
        'risk_focus': ['compliance'],
    },
    'legal_support': {
        'path': 'Legal support documents',
        'description': 'Legal correspondence and supporting documentation',
        'document_types': ['Legal Memos', 'Correspondence'],
        'risk_focus': ['legal'],
    },
    'relationship_governance': {
        'path': 'Relationship and governance docs',
        'description': 'Governance frameworks and relationship management',
        'document_types': ['Governance Frameworks', 'Escalation Matrices'],
        'risk_focus': ['operational', 'compliance'],
    }
}

print(f"Contract categories defined: {len(CONTRACT_CATEGORIES)}")
for cat, info in CONTRACT_CATEGORIES.items():
    print(f"  - {cat}: {info['description'][:50]}...")

## 1.6 Data Discovery

**TODO:** Implement the data discovery function.

### Hints:
- Use `Path.rglob('*')` to recursively find files
- Check file suffixes with `item.suffix.lower()`
- Create spans for each category scan

In [ ]:
# ============================================================================
# WEEK 1.6: DATA DISCOVERY FUNCTION
# ============================================================================

def discover_contract_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all contract documents organized by category.

    Args:
        data_dir: Path to the root data directory

    Returns:
        Dictionary mapping categories to lists of file paths

    TODO: Implement this function

    Steps:
    1. Create a trace for data discovery
    2. Initialize discovered dict with empty lists for each category
    3. For each category, scan the appropriate subdirectory
    4. Find all .docx, .pdf, .xlsx files
    5. Log results to Langfuse
    """

    # TODO: Create trace for data discovery
    # YOUR CODE HERE:
    trace = None


    # TODO: Initialize discovered dictionary
    # Hint: discovered = {cat: [] for cat in CONTRACT_CATEGORIES.keys()}

    # YOUR CODE HERE:
    discovered = {}


    # TODO: Check if data directory exists
    # YOUR CODE HERE:


    # TODO: Scan each category
    # Hint: Loop through CONTRACT_CATEGORIES, build path, use rglob
    for category, info in CONTRACT_CATEGORIES.items():
        # YOUR CODE HERE:
        pass


    # TODO: Calculate and log totals
    # YOUR CODE HERE:


    return discovered

# Set up data path
if IN_COLAB:
    DATA_DIR = Path('/content/data')
else:
    DATA_DIR = Path('../data')

# Run discovery
contract_files = discover_contract_data(DATA_DIR)

print("\nDiscovery Summary:")
total = 0
for cat, files in contract_files.items():
    if files:
        print(f"  {cat}: {len(files)} files")
        total += len(files)
print(f"\nTotal files discovered: {total}")

## Checkpoint: Week 1 Verification

Run this cell to verify your Week 1 implementation.

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 1 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Langfuse initialized
try:
    checks.append(("Langfuse initialized", langfuse is not None))
except:
    checks.append(("Langfuse initialized", False))

# Check 2: Session ID created
try:
    checks.append(("Session ID created", SESSION_ID is not None and len(SESSION_ID) > 10))
except:
    checks.append(("Session ID created", False))

# Check 3: OpenAI client ready
try:
    checks.append(("OpenAI client ready", openai_client is not None))
except:
    checks.append(("OpenAI client ready", False))

# Check 4: traced_embedding works
try:
    test_emb = traced_embedding("test", "test-validation")
    checks.append(("traced_embedding() works", test_emb is not None and len(test_emb) == 1536))
except Exception as e:
    checks.append(("traced_embedding() works", False))
    print(f"  Error: {e}")

# Check 5: traced_completion works
try:
    test_comp = traced_completion("Say 'hello'", "test-validation")
    checks.append(("traced_completion() works", test_comp is not None and len(test_comp) > 0))
except Exception as e:
    checks.append(("traced_completion() works", False))
    print(f"  Error: {e}")

# Check 6: Contract taxonomy
checks.append(("Contract taxonomy defined", len(CONTRACT_CATEGORIES) == 8))

# Check 7: Data discovery
try:
    checks.append(("Data discovery function works", callable(discover_contract_data)))
except:
    checks.append(("Data discovery function works", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 1 CHECKPOINTS PASSED! Ready for Week 2.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")
    print("\nHint: Check the error messages and compare with expected output.")

---

# WEEK 2: Document Processing & EDA

**Difficulty: Easy-Medium** - Implement document parsing methods.

---

## Learning Objectives

By the end of Week 2, you will:
- [ ] Load and parse DOCX contract files
- [ ] Extract text from tables
- [ ] Identify document sections
- [ ] Perform exploratory data analysis
- [ ] Extract contract entities

---

## 2.1 Document Processor Class

**TODO:** Implement the core document processing methods.

### Hints:
- Use `python-docx` library (already imported as `Document`)
- Paragraphs are in `doc.paragraphs`
- Tables are in `doc.tables`
- Use regex patterns to identify section headers

In [ ]:
# ============================================================================
# WEEK 2.1: DOCUMENT PROCESSOR CLASS
# ============================================================================

from docx import Document as DocxDocument
import re
import hashlib

class ContractDocumentProcessor:
    """
    Production-grade document processor for contract analysis.

    YOU MUST IMPLEMENT:
    - load_docx(): Load and parse a DOCX file
    - extract_tables(): Extract text from tables
    - extract_sections(): Identify document sections
    - process_category(): Process all documents in a category
    """

    def __init__(self):
        self.supported_formats = ['.docx']
        self.processed_docs = []

        # Section header patterns (PROVIDED)
        self.section_patterns = [
            r'^(\d+\.\s+[A-Z][A-Z\s]+)$',      # "1. DEFINITIONS"
            r'^(\d+\.\d+\s+.+)$',              # "1.1 Term"
            r'^(ARTICLE\s+[IVXLCDM]+)',          # "ARTICLE I"
            r'^(SECTION\s+\d+)',                # "SECTION 1"
            r'^(Schedule\s+[A-Z0-9]+)',          # "Schedule A"
            r'^(Exhibit\s+[A-Z0-9]+)',           # "Exhibit A"
        ]

    def load_docx(self, file_path: Path, trace_parent=None) -> Optional[Dict[str, Any]]:
        """
        Load and parse a DOCX contract file.

        Args:
            file_path: Path to the DOCX file
            trace_parent: Parent trace for nested spans

        Returns:
            Dictionary with parsed document data, or None on error

        TODO: Implement this method

        Steps:
        1. Create a span if trace_parent exists
        2. Open the DOCX file using DocxDocument(file_path)
        3. Extract all paragraph text
        4. Extract table text using extract_tables()
        5. Generate a unique doc_id
        6. Return dictionary with: doc_id, filename, full_text, table_text, sections
        """

        # TODO: Create span for tracing (if parent exists)
        span = trace_parent.span(name=f"load-{file_path.name}") if trace_parent else None

        try:
            # TODO: Load the DOCX file
            # Hint: doc = DocxDocument(file_path)

            # YOUR CODE HERE:
            doc = None


            # TODO: Extract paragraph text
            # Hint: Loop through doc.paragraphs and join their text

            # YOUR CODE HERE:
            full_text = ""


            # TODO: Extract table text
            # Hint: Use self.extract_tables(doc)

            # YOUR CODE HERE:
            table_text = ""


            # TODO: Generate unique doc_id
            # Hint: Use hashlib.md5 on filename

            # YOUR CODE HERE:
            doc_id = ""


            # TODO: Extract sections
            # Hint: Use self.extract_sections(full_text)

            # YOUR CODE HERE:
            sections = []


            # End span if exists
            if span:
                span.end(output={"status": "success", "text_length": len(full_text)})

            return {
                'doc_id': doc_id,
                'filename': file_path.name,
                'file_path': str(file_path),
                'full_text': full_text,
                'table_text': table_text,
                'sections': sections,
                'word_count': len(full_text.split())
            }

        except Exception as e:
            if span:
                span.end(output={"status": "error", "error": str(e)})
            print(f"Error loading {file_path}: {e}")
            return None

    def extract_tables(self, doc) -> str:
        """
        Extract text from all tables in a document.

        Args:
            doc: A python-docx Document object

        Returns:
            String containing all table text

        TODO: Implement this method

        Hints:
        - Tables are in doc.tables
        - Each table has .rows
        - Each row has .cells
        - Each cell has .text
        """

        # YOUR CODE HERE:
        table_texts = []

        # TODO: Loop through doc.tables
        # TODO: For each table, loop through rows
        # TODO: For each row, extract cell text
        # TODO: Join into formatted string

        return "\n".join(table_texts)

    def extract_sections(self, text: str) -> List[Dict[str, str]]:
        """
        Extract sections from document text using regex patterns.

        Args:
            text: Full document text

        Returns:
            List of dictionaries with section info

        TODO: Implement this method

        Hints:
        - Split text into lines
        - Check each line against self.section_patterns
        - Track section headers and their positions
        """

        # YOUR CODE HERE:
        sections = []

        # TODO: Split text into lines
        # TODO: For each line, check against section patterns
        # TODO: If match found, record section header

        return sections

print("ContractDocumentProcessor class defined")
print("TODO: Implement load_docx(), extract_tables(), extract_sections()")

### Expected Output for load_docx():
```python
{
    'doc_id': 'ABC123...',
    'filename': 'sample_msa.docx',
    'full_text': 'MASTER SERVICE AGREEMENT...',
    'table_text': 'Service | Price | Term...',
    'sections': [{'header': '1. DEFINITIONS', 'position': 0}, ...],
    'word_count': 5432
}
```

In [ ]:
# ============================================================================
# WEEK 2.1: PROCESS CATEGORY METHOD
# ============================================================================

def process_category(self, files: List[Path], category: str) -> List[Dict]:
    """
    Process all documents in a category with tracing.

    Args:
        files: List of file paths
        category: Category name

    Returns:
        List of processed document dictionaries

    TODO: Implement this method
    """

    # TODO: Create trace for this category
    trace = langfuse.trace(
        name=f"process-{category}",
        session_id=SESSION_ID,
        input={"category": category, "file_count": len(files)}
    )

    processed = []

    # TODO: Loop through files and process each one
    for file_path in files:
        # TODO: Check if file format is supported
        # TODO: Call load_docx with trace as parent
        # TODO: Add category to result
        # TODO: Append to processed list

        # YOUR CODE HERE:
        pass

    # Update trace with results
    trace.update(output={"processed_count": len(processed)})
    self.processed_docs.extend(processed)

    return processed

# Add method to class
ContractDocumentProcessor.process_category = process_category

print("process_category() method added")

## 2.2 Process All Documents

Run your document processor on all discovered files.

In [ ]:
# ============================================================================
# WEEK 2.2: PROCESS ALL DOCUMENTS
# ============================================================================

# Initialize processor
doc_processor = ContractDocumentProcessor()

print("PROCESSING CONTRACT DOCUMENTS")
print("=" * 60)

# Process all categories
all_contract_docs = {}
for category, files in contract_files.items():
    if files:
        all_contract_docs[category] = doc_processor.process_category(files, category)
        print(f"{category}: {len(all_contract_docs[category])} documents processed")

# Flatten all documents
all_docs_flat = []
for docs in all_contract_docs.values():
    all_docs_flat.extend(docs)

print(f"\n" + "=" * 60)
print(f"TOTAL DOCUMENTS PROCESSED: {len(all_docs_flat)}")

# If no real documents, create sample data
if not all_docs_flat:
    print("\nNo documents found - creating sample data for demonstration...")
    sample_contract_text = """
MASTER SERVICE AGREEMENT

This Master Service Agreement ("Agreement") is entered into as of January 1, 2024
between TechCorp Inc. ("Client") and ServicePro LLC ("Provider").

1. DEFINITIONS
   "Services" means the professional services described in any Statement of Work.
   "Confidential Information" means any non-public information disclosed by either party.

2. SERVICES AND DELIVERABLES
   Provider shall perform the Services described in each Statement of Work.
   All deliverables shall meet the specifications set forth in the applicable SOW.

3. PAYMENT TERMS
   Client shall pay Provider within Net 30 days of invoice receipt.
   Late payments shall accrue interest at 1.5% per month.
   Total contract value: $500,000 over 24 months.

4. LIABILITY
   Provider's total liability shall not exceed the fees paid in the prior 12 months.
   Neither party shall be liable for consequential, indirect, or punitive damages.

5. INDEMNIFICATION
   Each party shall indemnify the other against third-party claims arising from
   their negligence or willful misconduct.

6. TERMINATION
   Either party may terminate with 30 days written notice.
   Client may terminate for cause immediately upon material breach.
"""

    all_docs_flat = [{
        'doc_id': 'SAMPLE-001',
        'filename': 'sample_msa.docx',
        'category': 'master_agreements',
        'full_text': sample_contract_text,
        'table_text': '',
        'sections': [{'header': '1. DEFINITIONS', 'position': 0}],
        'word_count': len(sample_contract_text.split())
    }]

    print(f"Sample document created with {all_docs_flat[0]['word_count']} words")

langfuse.flush()

## 2.3 Exploratory Data Analysis (EDA)

**TODO:** Create visualizations to understand the contract corpus.

### Hints:
- Use matplotlib/seaborn for visualizations
- Create charts for: category distribution, word counts, section counts

In [ ]:
# ============================================================================
# WEEK 2.3: EXPLORATORY DATA ANALYSIS
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# TODO: Create a DataFrame from all_docs_flat
# Hint: df = pd.DataFrame(all_docs_flat)

# YOUR CODE HERE:
df = None


# TODO: Create visualization 1 - Category Distribution
# Hint: Use df['category'].value_counts() and plt.bar()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# YOUR CODE HERE for axes[0, 0]:
# Category distribution bar chart


# TODO: Create visualization 2 - Word Count Distribution
# Hint: Use df['word_count'] and plt.hist()

# YOUR CODE HERE for axes[0, 1]:
# Word count histogram


# TODO: Create visualization 3 - Word Count by Category
# Hint: Use seaborn boxplot

# YOUR CODE HERE for axes[1, 0]:
# Box plot of word counts by category


# TODO: Create visualization 4 - Summary Statistics
# Hint: Create a table or text summary

# YOUR CODE HERE for axes[1, 1]:
# Summary statistics display


plt.tight_layout()
plt.savefig('contract_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nEDA visualizations saved to contract_eda.png")

## 2.4 Contract Entity Extraction

**TODO:** Implement entity extraction using regex patterns.

### Hints:
- Use `re.findall()` for pattern matching
- Common entities: parties, dates, monetary values, percentages
- Risk indicators: liability terms, termination clauses

In [ ]:
# ============================================================================
# WEEK 2.4: CONTRACT ENTITY EXTRACTOR
# ============================================================================

class ContractEntityExtractor:
    """
    Extract contract-specific entities using pattern matching.

    TODO: Implement extract_entities() and assess_risk_indicators()
    """

    def __init__(self):
        # Extraction patterns (PROVIDED)
        self.patterns = {
            'parties': r'(?:between|by and between)\s+([A-Z][A-Za-z\s]+(?:Inc\.|LLC|Ltd\.|Corp\.)?)',
            'effective_date': r'(?:effective\s+(?:as\s+of\s+)?(?:date)?:?\s*)(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|[A-Z][a-z]+\s+\d{1,2},?\s+\d{4})',
            'monetary_values': r'\$([\d,]+(?:\.\d{2})?)',
            'percentages': r'(\d+(?:\.\d+)?\s*%)',
            'durations': r'(\d+)\s*(?:years?|months?|days?)',
            'notice_periods': r'(?:notice\s+(?:period)?\s*(?:of)?\s*)(\d+)\s*(?:days?)',
        }

        # Risk indicators (PROVIDED)
        self.risk_indicators = {
            'high_risk': [
                'unlimited liability', 'indemnify', 'consequential damages',
                'punitive damages', 'sole discretion', 'automatic renewal'
            ],
            'medium_risk': [
                'liability cap', 'limitation of liability', 'force majeure',
                'material breach', 'termination for convenience'
            ],
            'low_risk': [
                'mutual indemnification', 'standard terms', 'annual review'
            ]
        }

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """
        Extract all entities from contract text.

        Args:
            text: Contract text

        Returns:
            Dictionary mapping entity types to found values

        TODO: Implement this method

        Hints:
        - Loop through self.patterns
        - Use re.findall(pattern, text, re.IGNORECASE)
        - Store results in dictionary
        """

        # YOUR CODE HERE:
        entities = {}

        # TODO: For each pattern, find all matches


        return entities

    def assess_risk_indicators(self, text: str) -> Dict[str, List[str]]:
        """
        Identify risk indicators in contract text.

        Args:
            text: Contract text

        Returns:
            Dictionary with found risk indicators by level

        TODO: Implement this method

        Hints:
        - Loop through self.risk_indicators
        - Check if each indicator phrase is in text.lower()
        - Track which indicators were found
        """

        # YOUR CODE HERE:
        found_risks = {'high_risk': [], 'medium_risk': [], 'low_risk': []}

        # TODO: Check each risk indicator


        return found_risks

# Initialize extractor
entity_extractor = ContractEntityExtractor()

print("ContractEntityExtractor class defined")
print("TODO: Implement extract_entities() and assess_risk_indicators()")

In [ ]:
# ============================================================================
# TEST ENTITY EXTRACTION
# ============================================================================

# Test on sample document
if all_docs_flat:
    test_doc = all_docs_flat[0]
    print(f"Testing entity extraction on: {test_doc['filename']}")
    print("=" * 60)

    # Extract entities
    entities = entity_extractor.extract_entities(test_doc['full_text'])
    print("\nExtracted Entities:")
    for entity_type, values in entities.items():
        if values:
            print(f"  {entity_type}: {values[:3]}...")  # Show first 3

    # Assess risk
    risks = entity_extractor.assess_risk_indicators(test_doc['full_text'])
    print("\nRisk Indicators Found:")
    for level, indicators in risks.items():
        if indicators:
            print(f"  {level}: {indicators}")

## Checkpoint: Week 2 Verification

In [ ]:
# ============================================================================
# WEEK 2 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 2 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Document processor exists
checks.append(("DocumentProcessor initialized", doc_processor is not None))

# Check 2: load_docx method works
try:
    # Test with sample text (doesn't need real file)
    test_result = doc_processor.load_docx is not None
    checks.append(("load_docx() method exists", test_result))
except:
    checks.append(("load_docx() method exists", False))

# Check 3: Documents processed
checks.append(("Documents processed", len(all_docs_flat) > 0))

# Check 4: Entity extractor works
try:
    test_entities = entity_extractor.extract_entities("Contract value is $100,000")
    has_monetary = 'monetary_values' in test_entities
    checks.append(("Entity extraction works", has_monetary))
except Exception as e:
    checks.append(("Entity extraction works", False))
    print(f"  Error: {e}")

# Check 5: Risk assessment works
try:
    test_risks = entity_extractor.assess_risk_indicators("unlimited liability clause")
    has_high = len(test_risks.get('high_risk', [])) > 0
    checks.append(("Risk assessment works", has_high))
except Exception as e:
    checks.append(("Risk assessment works", False))
    print(f"  Error: {e}")

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 2 CHECKPOINTS PASSED! Ready for Week 3.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")

---

# WEEK 3: Vector Store & Embeddings

**Difficulty: Medium** - Implement semantic search functionality.

---

## Learning Objectives

By the end of Week 3, you will:
- [ ] Initialize ChromaDB for vector storage
- [ ] Implement document indexing with embeddings
- [ ] Create semantic search functions
- [ ] Understand similarity scoring

---

## 3.1 ChromaDB Vector Store Class

**TODO:** Implement the vector store with search capabilities.

### Hints:
- Use `chromadb.PersistentClient` for storage
- Create collections with `client.get_or_create_collection()`
- Use your `traced_embedding()` function for embeddings
- ChromaDB's `query()` method returns results with distances

In [ ]:
# ============================================================================
# WEEK 3.1: CHROMADB VECTOR STORE
# ============================================================================

import chromadb

class ContractVectorStore:
    """
    ChromaDB-based vector store with Langfuse observability.

    TODO: Implement these methods:
    - create_collection(): Create or get a collection
    - add_documents(): Index documents with embeddings
    - search(): Semantic search across documents
    - search_with_filter(): Search with metadata filters
    """

    def __init__(self, persist_directory: str = "./chroma_contracts_db"):
        """
        Initialize ChromaDB with persistent storage.

        TODO: Complete the initialization
        """

        # TODO: Create persistent client
        # Hint: self.client = chromadb.PersistentClient(path=persist_directory)

        # YOUR CODE HERE:
        self.client = None


        self.embedding_model = "text-embedding-3-small"
        self.collections = {}

        # Log initialization to Langfuse
        langfuse.trace(
            name="vectorstore-init",
            session_id=SESSION_ID,
            input={"persist_directory": persist_directory}
        )

        print(f"ChromaDB initialized at: {persist_directory}")

    def create_collection(self, name: str):
        """
        Create or get a collection.

        Args:
            name: Collection name

        Returns:
            ChromaDB collection

        TODO: Implement this method

        Hints:
        - Use self.client.get_or_create_collection(name=name)
        - Store in self.collections[name]
        """

        # YOUR CODE HERE:
        collection = None


        self.collections[name] = collection
        return collection

    def add_documents(self, collection_name: str, documents: List[Dict]):
        """
        Index documents into a collection.

        Args:
            collection_name: Target collection name
            documents: List of document dictionaries

        TODO: Implement this method

        Steps:
        1. Get or create collection
        2. For each document:
           a. Generate embedding using traced_embedding()
           b. Prepare metadata (filename, category, word_count)
           c. Add to collection with doc_id as ID
        3. Log progress

        Hints:
        - Use collection.add(ids=[...], embeddings=[...], documents=[...], metadatas=[...])
        - Truncate text before embedding (max ~8000 chars)
        """

        # Create trace for indexing
        trace = langfuse.trace(
            name=f"index-{collection_name}",
            session_id=SESSION_ID,
            input={"doc_count": len(documents)}
        )

        # Get or create collection
        collection = self.create_collection(collection_name)

        # TODO: Process and add documents
        ids = []
        embeddings = []
        docs = []
        metadatas = []

        for i, doc in enumerate(documents):
            # TODO: Generate embedding
            # Hint: Use traced_embedding(doc['full_text'][:8000], f"embed-{doc['doc_id']}")

            # YOUR CODE HERE:


            # TODO: Prepare metadata
            # YOUR CODE HERE:


            # Progress indicator
            if (i + 1) % 5 == 0:
                print(f"  Indexed {i + 1}/{len(documents)} documents...")

        # TODO: Add all to collection
        # Hint: collection.add(ids=ids, embeddings=embeddings, documents=docs, metadatas=metadatas)

        # YOUR CODE HERE:


        trace.update(output={"indexed": len(documents)})
        print(f"Indexed {len(documents)} documents into '{collection_name}'")

    def search(self, collection_name: str, query: str, n_results: int = 5) -> List[Dict]:
        """
        Semantic search across documents.

        Args:
            collection_name: Collection to search
            query: Search query text
            n_results: Number of results to return

        Returns:
            List of result dictionaries with document, metadata, and score

        TODO: Implement this method

        Steps:
        1. Generate embedding for query
        2. Query collection using query_embeddings
        3. Format results with scores

        Hints:
        - Use collection.query(query_embeddings=[query_embedding], n_results=n_results)
        - Results contain 'documents', 'metadatas', 'distances'
        - Convert distance to similarity: similarity = 1 - distance (for cosine)
        """

        # Create trace for search
        trace = langfuse.trace(
            name="semantic-search",
            session_id=SESSION_ID,
            input={"query": query, "collection": collection_name}
        )

        # TODO: Get collection
        # YOUR CODE HERE:
        collection = None


        # TODO: Generate query embedding
        # YOUR CODE HERE:
        query_embedding = None


        # TODO: Query collection
        # YOUR CODE HERE:
        results = None


        # TODO: Format results
        formatted = []
        # Hint: Loop through results and create dict with 'document', 'metadata', 'similarity'

        # YOUR CODE HERE:


        trace.update(output={"results_count": len(formatted)})
        return formatted

print("ContractVectorStore class defined")
print("TODO: Implement create_collection(), add_documents(), search()")

### Expected Output for search():
```python
[
    {
        'document': 'Provider shall perform the Services...',
        'metadata': {'filename': 'sample_msa.docx', 'category': 'master_agreements'},
        'similarity': 0.87
    },
    ...
]
```

In [ ]:
# ============================================================================
# WEEK 3.2: INITIALIZE AND INDEX DOCUMENTS
# ============================================================================

# Initialize vector store
vector_store = ContractVectorStore()

print("\n" + "=" * 60)
print("INDEXING CONTRACT DOCUMENTS")
print("=" * 60)

# Index all documents
if all_docs_flat:
    vector_store.add_documents("contracts_all", all_docs_flat)
else:
    print("No documents to index!")

langfuse.flush()
print("\nIndexing complete!")

## 3.3 Semantic Search Examples

Test your search implementation with various queries.

In [ ]:
# ============================================================================
# WEEK 3.3: SEMANTIC SEARCH EXAMPLES
# ============================================================================

print("SEMANTIC SEARCH EXAMPLES")
print("=" * 60)

# Test queries
test_queries = [
    "liability limitations and caps",
    "payment terms and invoicing",
    "termination for convenience",
    "intellectual property rights",
    "confidentiality obligations"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 40)

    results = vector_store.search("contracts_all", query, n_results=3)

    for i, result in enumerate(results, 1):
        print(f"  {i}. [{result.get('similarity', 0):.3f}] {result.get('metadata', {}).get('filename', 'unknown')}")
        # Show snippet
        doc_text = result.get('document', '')[:150]
        print(f"     '{doc_text}...'")

langfuse.flush()

## 3.4 Search with Metadata Filters (BONUS)

**BONUS TODO:** Implement filtered search.

### Hints:
- ChromaDB supports `where` parameter for filtering
- Filter format: `{"category": "master_agreements"}`

In [ ]:
# ============================================================================
# WEEK 3.4: FILTERED SEARCH (BONUS)
# ============================================================================

def search_with_filter(self, collection_name: str, query: str,
                       filter_dict: Dict = None, n_results: int = 5) -> List[Dict]:
    """
    Semantic search with metadata filtering.

    Args:
        collection_name: Collection to search
        query: Search query
        filter_dict: Metadata filter (e.g., {"category": "master_agreements"})
        n_results: Number of results

    Returns:
        Filtered search results

    BONUS TODO: Implement this method

    Hints:
    - Add `where=filter_dict` parameter to collection.query()
    """

    # YOUR CODE HERE:
    pass

# Add method to class
ContractVectorStore.search_with_filter = search_with_filter

print("search_with_filter() method added (BONUS)")

## Checkpoint: Week 3 Verification

In [ ]:
# ============================================================================
# WEEK 3 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 3 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Vector store initialized
checks.append(("VectorStore initialized", vector_store is not None))

# Check 2: ChromaDB client created
try:
    checks.append(("ChromaDB client created", vector_store.client is not None))
except:
    checks.append(("ChromaDB client created", False))

# Check 3: Collection created
try:
    has_collection = "contracts_all" in vector_store.collections
    checks.append(("Collection created", has_collection))
except:
    checks.append(("Collection created", False))

# Check 4: Search returns results
try:
    results = vector_store.search("contracts_all", "liability", n_results=1)
    checks.append(("Search returns results", len(results) > 0))
except Exception as e:
    checks.append(("Search returns results", False))
    print(f"  Error: {e}")

# Check 5: Results have correct structure
try:
    if results:
        has_doc = 'document' in results[0]
        has_meta = 'metadata' in results[0]
        has_sim = 'similarity' in results[0]
        checks.append(("Result structure correct", has_doc and has_meta and has_sim))
    else:
        checks.append(("Result structure correct", False))
except:
    checks.append(("Result structure correct", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 3 CHECKPOINTS PASSED! Ready for Week 4.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")

---

# WEEK 4: Single Agent Design

**Difficulty: Medium-Hard** - Build specialized risk assessment agents.

---

## Learning Objectives

By the end of Week 4, you will:
- [ ] Define Pydantic models for structured outputs
- [ ] Create LangChain prompt templates
- [ ] Implement specialized risk agents
- [ ] Integrate Langfuse callback handlers

---

## Key Concept: Agent Architecture

Each agent follows this pattern:
```
Prompt Template -> LLM -> Output Parser -> Structured Result
       |              |           |
       v              v           v
   (Expert persona)  (GPT-4)   (Pydantic model)
```

All calls are traced through Langfuse for observability.

## 4.1 Pydantic Models for Structured Output

**TODO:** Complete the Pydantic models for each risk type.

### Hints:
- Use `Field()` with descriptions to guide the LLM
- Include `confidence` field (0.0-1.0) in all models
- Think about what fields each risk type needs

In [ ]:
# ============================================================================
# WEEK 4.1: PYDANTIC MODELS
# ============================================================================

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler

class LegalRiskAssessment(BaseModel):
    """
    Structured output for legal risk analysis.

    TODO: Complete this model with appropriate fields

    Required fields:
    - contract_type: str
    - risk_level: str (LOW, MEDIUM, HIGH, CRITICAL)
    - liability_exposure: str
    - ip_risks: List[str]
    - indemnification_issues: List[str]
    - compliance_gaps: List[str]
    - recommended_changes: List[str]
    - confidence: float (0.0 to 1.0)
    """

    # PROVIDED - contract type
    contract_type: str = Field(description="Type of contract (MSA, NDA, SOW, etc.)")

    # TODO: Add risk_level field
    # YOUR CODE HERE:
    risk_level: str = Field(description="Overall risk level: LOW, MEDIUM, HIGH, CRITICAL")

    # TODO: Add liability_exposure field
    # YOUR CODE HERE:


    # TODO: Add ip_risks field (List[str])
    # YOUR CODE HERE:


    # TODO: Add indemnification_issues field (List[str])
    # YOUR CODE HERE:


    # TODO: Add compliance_gaps field (List[str])
    # YOUR CODE HERE:


    # TODO: Add recommended_changes field (List[str])
    # YOUR CODE HERE:


    # TODO: Add confidence field (float, 0.0 to 1.0)
    # Hint: Use Field(ge=0.0, le=1.0, description="...")
    # YOUR CODE HERE:


print("LegalRiskAssessment model defined")
print(f"Fields: {list(LegalRiskAssessment.model_fields.keys())}")

In [ ]:
# ============================================================================
# FINANCIAL RISK ASSESSMENT MODEL
# ============================================================================

class FinancialRiskAssessment(BaseModel):
    """
    Structured output for financial risk analysis.

    TODO: Complete this model

    Required fields:
    - total_contract_value: str
    - payment_terms: str
    - pricing_risks: List[str]
    - penalty_clauses: List[str]
    - cash_flow_impact: str
    - financial_exposure: str
    - risk_level: str
    - confidence: float
    """

    # TODO: Add all fields
    # YOUR CODE HERE:
    total_contract_value: str = Field(description="Total estimated contract value")
    payment_terms: str = Field(description="Summary of payment terms")

    # Add remaining fields...


print("FinancialRiskAssessment model defined")

In [ ]:
# ============================================================================
# OPERATIONAL RISK ASSESSMENT MODEL
# ============================================================================

class OperationalRiskAssessment(BaseModel):
    """
    Structured output for operational risk analysis.

    TODO: Complete this model

    Required fields:
    - service_scope: str
    - sla_requirements: List[str]
    - delivery_risks: List[str]
    - resource_requirements: str
    - dependency_risks: List[str]
    - mitigation_strategies: List[str]
    - risk_level: str
    - confidence: float
    """

    # TODO: Add all fields
    # YOUR CODE HERE:
    service_scope: str = Field(description="Summary of service scope")

    # Add remaining fields...


print("OperationalRiskAssessment model defined")

## 4.2 Langfuse Callback Handler Factory

**PROVIDED** - This helper creates traced handlers for LangChain.

In [ ]:
# ============================================================================
# WEEK 4.2: LANGFUSE CALLBACK HANDLER (PROVIDED)
# ============================================================================

def get_langfuse_handler(trace_name: str, tags: List[str] = None) -> LangfuseCallbackHandler:
    """
    Create a Langfuse callback handler for LangChain.

    This traces:
    - Prompt formatting
    - LLM calls (input/output/tokens)
    - Output parsing
    - Errors
    """
    return LangfuseCallbackHandler(
        secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
        public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
        host=os.environ.get('LANGFUSE_HOST'),
        session_id=SESSION_ID,
        trace_name=trace_name,
        tags=tags or []
    )

print("get_langfuse_handler() factory defined")

## 4.3 Legal Risk Agent

**TODO:** Implement the Legal Risk Agent from scratch.

### Hints:
- Create a `ChatPromptTemplate` with system and human messages
- Use `PydanticOutputParser` with your model
- Chain: prompt | llm | parser
- Include `{format_instructions}` in the prompt

In [ ]:
# ============================================================================
# WEEK 4.3: LEGAL RISK AGENT
# ============================================================================

class LegalRiskAgent:
    """
    Legal Risk Assessment Agent with Langfuse tracing.

    TODO: Implement the __init__ and analyze methods

    Architecture:
    1. Prompt template with expert legal persona
    2. ChatOpenAI LLM
    3. PydanticOutputParser for structured output
    4. LangChain LCEL chain
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model

        # TODO: Create output parser
        # Hint: self.parser = PydanticOutputParser(pydantic_object=LegalRiskAssessment)

        # YOUR CODE HERE:
        self.parser = None


        # TODO: Create prompt template
        # Hint: Use ChatPromptTemplate.from_messages([...])
        # Include a system message with expert persona
        # Include {format_instructions} placeholder
        # Include {contract_text} placeholder

        # YOUR CODE HERE:
        self.prompt = None

        # Example structure (uncomment and complete):
        # self.prompt = ChatPromptTemplate.from_messages([
        #     ("system", """You are an expert contract attorney...
        #
        #     {format_instructions}"""),
        #     ("human", """Analyze the following contract for legal risks:
        #
        #     {contract_text}
        #
        #     Provide a comprehensive legal risk assessment.""")
        # ])

    def analyze(self, contract_text: str, doc_id: str = "unknown") -> LegalRiskAssessment:
        """
        Analyze contract for legal risks.

        Args:
            contract_text: Contract text to analyze
            doc_id: Document identifier for tracing

        Returns:
            LegalRiskAssessment structured output

        TODO: Implement this method

        Steps:
        1. Create Langfuse handler using get_langfuse_handler()
        2. Create ChatOpenAI with handler as callback
        3. Build LCEL chain: self.prompt | llm | self.parser
        4. Invoke chain with contract_text and format_instructions
        """

        # TODO: Create traced handler
        # YOUR CODE HERE:
        handler = None


        # TODO: Create LLM with callback
        # Hint: llm = ChatOpenAI(model=self.model, temperature=0, callbacks=[handler])

        # YOUR CODE HERE:
        llm = None


        # TODO: Build chain
        # Hint: chain = self.prompt | llm | self.parser

        # YOUR CODE HERE:
        chain = None


        # TODO: Invoke chain
        # Hint: return chain.invoke({"contract_text": ..., "format_instructions": ...})

        # YOUR CODE HERE:
        result = None


        return result

print("LegalRiskAgent class defined")
print("TODO: Implement __init__() and analyze()")

In [ ]:
# ============================================================================
# WEEK 4.4: FINANCIAL RISK AGENT
# ============================================================================

class FinancialRiskAgent:
    """
    Financial Risk Assessment Agent.

    TODO: Implement similar to LegalRiskAgent but with financial focus

    Expert persona should focus on:
    - Pricing and contract value
    - Payment terms and cash flow
    - Financial penalties and exposure
    - Cost escalation risks
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model

        # TODO: Create parser for FinancialRiskAssessment
        # YOUR CODE HERE:
        self.parser = None


        # TODO: Create prompt with financial expert persona
        # YOUR CODE HERE:
        self.prompt = None


    def analyze(self, contract_text: str, doc_id: str = "unknown") -> FinancialRiskAssessment:
        """Analyze contract for financial risks."""

        # TODO: Implement (same pattern as LegalRiskAgent)
        # YOUR CODE HERE:
        pass

print("FinancialRiskAgent class defined")

In [ ]:
# ============================================================================
# WEEK 4.5: OPERATIONAL RISK AGENT
# ============================================================================

class OperationalRiskAgent:
    """
    Operational Risk Assessment Agent.

    TODO: Implement similar to other agents but with operational focus

    Expert persona should focus on:
    - Service scope and deliverables
    - SLA requirements
    - Delivery risks and timelines
    - Resource and dependency management
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model

        # TODO: Create parser for OperationalRiskAssessment
        # YOUR CODE HERE:
        self.parser = None


        # TODO: Create prompt with operations expert persona
        # YOUR CODE HERE:
        self.prompt = None


    def analyze(self, contract_text: str, doc_id: str = "unknown") -> OperationalRiskAssessment:
        """Analyze contract for operational risks."""

        # TODO: Implement
        # YOUR CODE HERE:
        pass

print("OperationalRiskAgent class defined")

In [ ]:
# ============================================================================
# INITIALIZE ALL AGENTS
# ============================================================================

# Create agent instances
legal_agent = LegalRiskAgent()
financial_agent = FinancialRiskAgent()
operational_agent = OperationalRiskAgent()

print("Risk Assessment Agents Initialized:")
print("  - LegalRiskAgent")
print("  - FinancialRiskAgent")
print("  - OperationalRiskAgent")

## 4.6 Test Your Agents

Run each agent and verify the structured outputs.

In [ ]:
# ============================================================================
# WEEK 4.6: TEST AGENTS
# ============================================================================

print("TESTING RISK ASSESSMENT AGENTS")
print("=" * 60)

# Get test contract text
if all_docs_flat:
    test_text = all_docs_flat[0]['full_text']
    test_id = all_docs_flat[0]['doc_id']
else:
    test_text = "Sample contract with liability and payment terms."
    test_id = "TEST-001"

print(f"Testing on document: {test_id}")
print(f"Text length: {len(test_text)} characters")

# Test Legal Agent
print("\n" + "-" * 40)
print("LEGAL RISK AGENT")
print("-" * 40)
try:
    legal_result = legal_agent.analyze(test_text[:8000], test_id)
    print(f"Risk Level: {legal_result.risk_level}")
    print(f"Contract Type: {legal_result.contract_type}")
    print(f"Confidence: {legal_result.confidence}")
except Exception as e:
    print(f"Error: {e}")
    legal_result = None

# Test Financial Agent
print("\n" + "-" * 40)
print("FINANCIAL RISK AGENT")
print("-" * 40)
try:
    financial_result = financial_agent.analyze(test_text[:8000], test_id)
    print(f"Risk Level: {financial_result.risk_level}")
    print(f"Contract Value: {financial_result.total_contract_value}")
    print(f"Confidence: {financial_result.confidence}")
except Exception as e:
    print(f"Error: {e}")
    financial_result = None

# Test Operational Agent
print("\n" + "-" * 40)
print("OPERATIONAL RISK AGENT")
print("-" * 40)
try:
    operational_result = operational_agent.analyze(test_text[:8000], test_id)
    print(f"Risk Level: {operational_result.risk_level}")
    print(f"Service Scope: {operational_result.service_scope[:100]}...")
    print(f"Confidence: {operational_result.confidence}")
except Exception as e:
    print(f"Error: {e}")
    operational_result = None

langfuse.flush()
print("\nAll agent traces sent to Langfuse!")

## Checkpoint: Week 4 Verification

In [ ]:
# ============================================================================
# WEEK 4 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 4 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Pydantic models defined
try:
    legal_fields = list(LegalRiskAssessment.model_fields.keys())
    has_required = 'risk_level' in legal_fields and 'confidence' in legal_fields
    checks.append(("Pydantic models complete", has_required and len(legal_fields) >= 5))
except:
    checks.append(("Pydantic models complete", False))

# Check 2: Agents initialized
checks.append(("Legal agent initialized", legal_agent is not None))
checks.append(("Financial agent initialized", financial_agent is not None))
checks.append(("Operational agent initialized", operational_agent is not None))

# Check 3: Legal agent works
try:
    checks.append(("Legal agent produces output", legal_result is not None))
except:
    checks.append(("Legal agent produces output", False))

# Check 4: Financial agent works
try:
    checks.append(("Financial agent produces output", financial_result is not None))
except:
    checks.append(("Financial agent produces output", False))

# Check 5: Operational agent works
try:
    checks.append(("Operational agent produces output", operational_result is not None))
except:
    checks.append(("Operational agent produces output", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 4 CHECKPOINTS PASSED! Ready for Week 5.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")

---

# WEEK 5: Multi-Agent Orchestration

**Difficulty: Hard** - Coordinate multiple agents in parallel.

---

## Learning Objectives

By the end of Week 5, you will:
- [ ] Implement parallel agent execution
- [ ] Add retry logic with exponential backoff
- [ ] Build consensus across agent outputs
- [ ] Calculate composite risk scores

---

## Key Concept: Orchestration Patterns

```
                    +----------------+
                    |  Orchestrator  |
                    +----------------+
                           |
          +----------------+----------------+
          |                |                |
          v                v                v
    +-----------+    +-----------+    +-----------+
    |   Legal   |    | Financial |    |Operational|
    |   Agent   |    |   Agent   |    |   Agent   |
    +-----------+    +-----------+    +-----------+
          |                |                |
          +----------------+----------------+
                           |
                           v
                    +----------------+
                    |   Consensus    |
                    +----------------+
```

Agents run in parallel using ThreadPoolExecutor, with retry logic for resilience.

## 5.1 Multi-Agent Orchestrator

**TODO:** Implement the orchestrator class.

### Hints:
- Use `concurrent.futures.ThreadPoolExecutor` for parallel execution
- Use `tenacity` library for retry logic
- Implement weighted scoring for composite risk
- Log all operations to Langfuse

In [ ]:
# ============================================================================
# WEEK 5.1: MULTI-AGENT ORCHESTRATOR
# ============================================================================

from tenacity import retry, stop_after_attempt, wait_exponential
import concurrent.futures

class RiskDimension(Enum):
    """Risk dimensions for orchestration."""
    LEGAL = "legal"
    FINANCIAL = "financial"
    OPERATIONAL = "operational"

class ContractRiskOrchestrator:
    """
    Multi-agent orchestrator with parallel execution.

    TODO: Implement these methods:
    - _execute_agent(): Execute single agent with retry
    - analyze_parallel(): Run all agents in parallel
    - _build_consensus(): Aggregate results
    - _calculate_composite_score(): Weighted risk score
    """

    def __init__(self):
        self.legal_agent = legal_agent
        self.financial_agent = financial_agent
        self.operational_agent = operational_agent
        self.vector_store = vector_store

        # Risk weights (provided)
        self.risk_weights = {
            RiskDimension.LEGAL: 0.4,       # Legal carries most weight
            RiskDimension.FINANCIAL: 0.35,  # Financial is second
            RiskDimension.OPERATIONAL: 0.25 # Operational is third
        }

        # Risk level scores (provided)
        self.risk_scores = {
            'LOW': 1,
            'MEDIUM': 2,
            'HIGH': 3,
            'CRITICAL': 4
        }

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _execute_agent(self, agent_type: RiskDimension, contract_text: str, doc_id: str) -> Dict:
        """
        Execute a single agent with retry logic.

        Args:
            agent_type: Which agent to run
            contract_text: Contract to analyze
            doc_id: Document identifier

        Returns:
            Dictionary with agent type and result

        TODO: Implement this method

        Steps:
        1. Based on agent_type, call the appropriate agent's analyze method
        2. Convert result to dictionary using model_dump()
        3. Return dict with 'agent' and 'result' keys

        Hints:
        - if agent_type == RiskDimension.LEGAL: use self.legal_agent
        - Use result.model_dump() to convert Pydantic to dict
        """

        try:
            # TODO: Select and run appropriate agent
            # YOUR CODE HERE:

            if agent_type == RiskDimension.LEGAL:
                result = None  # TODO: Call legal agent
            elif agent_type == RiskDimension.FINANCIAL:
                result = None  # TODO: Call financial agent
            elif agent_type == RiskDimension.OPERATIONAL:
                result = None  # TODO: Call operational agent
            else:
                raise ValueError(f"Unknown agent type: {agent_type}")

            return {
                'agent': agent_type.value,
                'result': result.model_dump() if result else None
            }

        except Exception as e:
            print(f"Agent {agent_type.value} failed: {e}")
            raise  # Re-raise for retry

print("ContractRiskOrchestrator class started")
print("TODO: Implement _execute_agent()")

In [ ]:
# ============================================================================
# WEEK 5.2: PARALLEL EXECUTION METHOD
# ============================================================================

def analyze_parallel(self, contract_text: str, doc_id: str) -> Dict:
    """
    Run all agents in parallel and build consensus.

    Args:
        contract_text: Contract to analyze
        doc_id: Document identifier

    Returns:
        Dictionary with all results, consensus, and composite score

    TODO: Implement this method

    Steps:
    1. Create Langfuse trace for orchestration
    2. Use ThreadPoolExecutor to run agents in parallel
    3. Collect results from all agents
    4. Build consensus
    5. Calculate composite score
    """

    # Create orchestration trace
    trace = langfuse.trace(
        name=f"orchestration-{doc_id}",
        session_id=SESSION_ID,
        input={"doc_id": doc_id, "text_length": len(contract_text)}
    )

    results = {}

    # TODO: Run agents in parallel using ThreadPoolExecutor
    # Hint: with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    #           futures = {executor.submit(self._execute_agent, agent_type, ...): agent_type
    #                      for agent_type in RiskDimension}

    # YOUR CODE HERE:
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        # TODO: Submit all agent tasks
        futures = {}


        # TODO: Collect results as they complete
        for future in concurrent.futures.as_completed(futures):
            # YOUR CODE HERE:
            pass

    # TODO: Build consensus
    # Hint: consensus = self._build_consensus(results)

    # YOUR CODE HERE:
    consensus = {}


    # TODO: Calculate composite score
    # Hint: composite_score = self._calculate_composite_score(results)

    # YOUR CODE HERE:
    composite_score = 0.0


    # Determine overall risk level
    if composite_score < 1.5:
        overall_risk = "LOW"
    elif composite_score < 2.5:
        overall_risk = "MEDIUM"
    elif composite_score < 3.5:
        overall_risk = "HIGH"
    else:
        overall_risk = "CRITICAL"

    # Update trace
    trace.update(output={
        "overall_risk": overall_risk,
        "composite_score": composite_score
    })

    return {
        'doc_id': doc_id,
        'results': results,
        'consensus': consensus,
        'composite_score': composite_score,
        'overall_risk': overall_risk
    }

# Add method to class
ContractRiskOrchestrator.analyze_parallel = analyze_parallel

print("analyze_parallel() method added")

In [ ]:
# ============================================================================
# WEEK 5.3: CONSENSUS BUILDING
# ============================================================================

def _build_consensus(self, results: Dict) -> Dict:
    """
    Build consensus from multiple agent results.

    Args:
        results: Dictionary of agent results

    Returns:
        Consensus dictionary with aggregated findings

    TODO: Implement this method

    Steps:
    1. Collect all risk levels
    2. Find common themes across agents
    3. Aggregate recommendations
    """

    # YOUR CODE HERE:
    consensus = {
        'risk_levels': {},
        'key_findings': [],
        'all_recommendations': []
    }

    # TODO: Extract risk level from each agent result
    for agent_name, result in results.items():
        if result and 'result' in result:
            # TODO: Get risk level
            pass

    # TODO: Aggregate recommendations from all agents


    return consensus

def _calculate_composite_score(self, results: Dict) -> float:
    """
    Calculate weighted composite risk score.

    Args:
        results: Dictionary of agent results

    Returns:
        Weighted composite score (1.0 to 4.0)

    TODO: Implement this method

    Formula:
    composite = sum(weight * risk_score) for each agent

    Hints:
    - Use self.risk_weights for weights
    - Use self.risk_scores to convert level to number
    - Handle missing results gracefully
    """

    # YOUR CODE HERE:
    total_weight = 0
    weighted_sum = 0

    # TODO: Calculate weighted sum
    for dimension in RiskDimension:
        # TODO: Get result for this dimension
        # TODO: Get risk level and convert to score
        # TODO: Apply weight
        pass

    # Avoid division by zero
    if total_weight == 0:
        return 2.5  # Default to medium

    return weighted_sum / total_weight

# Add methods to class
ContractRiskOrchestrator._build_consensus = _build_consensus
ContractRiskOrchestrator._calculate_composite_score = _calculate_composite_score

print("Consensus methods added")

In [ ]:
# ============================================================================
# INITIALIZE ORCHESTRATOR
# ============================================================================

# Create orchestrator
orchestrator = ContractRiskOrchestrator()

print("ContractRiskOrchestrator initialized")
print(f"  Risk weights: {orchestrator.risk_weights}")

## 5.2 Test Orchestration

Run the orchestrator on a contract and observe parallel execution.

In [ ]:
# ============================================================================
# WEEK 5.4: TEST ORCHESTRATION
# ============================================================================

print("MULTI-AGENT ORCHESTRATION TEST")
print("=" * 60)

# Get test contract
if all_docs_flat:
    test_doc = all_docs_flat[0]
    test_text = test_doc['full_text']
    test_id = test_doc['doc_id']
else:
    test_text = "Sample contract text"
    test_id = "TEST-001"

print(f"Analyzing document: {test_id}")
print(f"Text length: {len(test_text)} characters")
print("\nRunning agents in parallel...")
print("-" * 40)

# Run orchestration
try:
    orch_result = orchestrator.analyze_parallel(test_text[:8000], test_id)

    print("\nORCHESTRATION RESULTS")
    print("=" * 60)
    print(f"Overall Risk: {orch_result['overall_risk']}")
    print(f"Composite Score: {orch_result['composite_score']:.2f}")

    print("\nIndividual Agent Results:")
    for agent, result in orch_result['results'].items():
        if result and 'result' in result:
            risk = result['result'].get('risk_level', 'N/A')
            conf = result['result'].get('confidence', 0)
            print(f"  {agent}: {risk} (confidence: {conf:.2f})")

    print("\nConsensus:")
    print(f"  Risk Levels: {orch_result['consensus'].get('risk_levels', {})}")

except Exception as e:
    print(f"Orchestration failed: {e}")
    orch_result = None

langfuse.flush()
print("\nOrchestration trace sent to Langfuse!")

## Checkpoint: Week 5 Verification

In [ ]:
# ============================================================================
# WEEK 5 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 5 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Orchestrator initialized
checks.append(("Orchestrator initialized", orchestrator is not None))

# Check 2: Risk weights configured
try:
    has_weights = len(orchestrator.risk_weights) == 3
    checks.append(("Risk weights configured", has_weights))
except:
    checks.append(("Risk weights configured", False))

# Check 3: Orchestration produced results
try:
    checks.append(("Orchestration completed", orch_result is not None))
except:
    checks.append(("Orchestration completed", False))

# Check 4: All agents ran
try:
    if orch_result:
        agent_count = len(orch_result.get('results', {}))
        checks.append(("All 3 agents ran", agent_count == 3))
    else:
        checks.append(("All 3 agents ran", False))
except:
    checks.append(("All 3 agents ran", False))

# Check 5: Composite score calculated
try:
    if orch_result:
        score = orch_result.get('composite_score', 0)
        checks.append(("Composite score valid", 1.0 <= score <= 4.0))
    else:
        checks.append(("Composite score valid", False))
except:
    checks.append(("Composite score valid", False))

# Check 6: Overall risk determined
try:
    if orch_result:
        risk = orch_result.get('overall_risk', '')
        valid_risks = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
        checks.append(("Overall risk determined", risk in valid_risks))
    else:
        checks.append(("Overall risk determined", False))
except:
    checks.append(("Overall risk determined", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 5 CHECKPOINTS PASSED! Ready for Week 6.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")

---

# WEEK 6: Contract Knowledge Graph

**Difficulty: Medium** - Build and visualize contract relationships.

---

## Learning Objectives

By the end of Week 6, you will:
- [ ] Build a knowledge graph using NetworkX
- [ ] Model contract type hierarchies
- [ ] Add query methods for insights
- [ ] Create visualizations with matplotlib and PyVis

---

## Key Concept: Knowledge Graphs

A knowledge graph represents entities and their relationships:

```
(MSA) --[GOVERNS]--> (SOW)
(MSA) --[CONTAINS]--> (Liability Clause)
(Liability Clause) --[MITIGATES]--> (Financial Risk)
```

**Benefits for Contract Intelligence:**
- Understand document hierarchies
- Trace risk to source clauses
- Identify missing protections

## 6.1 Contract Knowledge Graph Class

**TODO:** Implement the knowledge graph with contract ontology.

### Hints:
- Use `networkx.DiGraph()` for directed graph
- Add nodes with `graph.add_node(id, type=..., color=...)`
- Add edges with `graph.add_edge(source, target, relation=...)`

In [ ]:
# ============================================================================
# WEEK 6.1: CONTRACT KNOWLEDGE GRAPH
# ============================================================================

import networkx as nx
from pyvis.network import Network
import matplotlib.pyplot as plt

class ContractKnowledgeGraph:
    """
    Knowledge graph for contract relationships.

    TODO: Implement:
    - _build_contract_ontology(): Create base graph structure
    - add_contract(): Add a contract document
    - get_risk_mitigations(): Query for mitigations
    - visualize_matplotlib(): Static visualization
    - visualize_interactive(): Interactive HTML visualization
    """

    def __init__(self):
        self.graph = nx.DiGraph()
        self._build_contract_ontology()

        # Log initialization
        langfuse.trace(
            name="knowledge-graph-init",
            session_id=SESSION_ID,
            input={"type": "contract-ontology"}
        )

    def _build_contract_ontology(self):
        """
        Build the foundational contract knowledge graph.

        TODO: Implement this method

        Steps:
        1. Add contract type hierarchy (MSA -> SOW, NDA, etc.)
        2. Add clause types (INDEMNIFICATION, LIABILITY, etc.)
        3. Add risk types and mitigations
        4. Connect with appropriate relationships

        Hints:
        - Use self.graph.add_node(name, type='...', color='#...')
        - Use self.graph.add_edge(source, target, relation='...')
        """

        # TODO: Define contract type hierarchy
        contract_hierarchy = {
            'MSA': ['SOW', 'NDA', 'RATE_CARD'],
            'SOW': ['INVOICE', 'SERVICE_REPORT', 'CHANGE_ORDER'],
            'NDA': ['CONFIDENTIAL_INFO'],
        }

        # TODO: Add contract type nodes and edges
        # YOUR CODE HERE:
        for contract_type, subordinates in contract_hierarchy.items():
            # Add parent node
            # Add child nodes
            # Add GOVERNS relationships
            pass

        # TODO: Define clause types
        clause_types = [
            ('INDEMNIFICATION', ['liability_exposure', 'third_party_claims']),
            ('LIABILITY_LIMITATION', ['financial_cap', 'damage_types']),
            ('CONFIDENTIALITY', ['data_protection', 'trade_secret']),
            ('TERMINATION', ['exit_rights', 'notice_requirements']),
            ('IP_RIGHTS', ['ownership', 'licensing']),
            ('PAYMENT_TERMS', ['billing_cycle', 'late_fees']),
        ]

        # TODO: Add clause nodes and connections
        # YOUR CODE HERE:
        for clause, concerns in clause_types:
            # Add clause node with type='CLAUSE_TYPE', color='#e74c3c'
            # Add concern nodes with type='CONCERN', color='#f39c12'
            # Add ADDRESSES relationships
            pass

        # TODO: Define risk mitigations
        risk_mitigations = {
            'UNLIMITED_LIABILITY': ['LIABILITY_LIMITATION'],
            'IP_INFRINGEMENT': ['IP_RIGHTS', 'INDEMNIFICATION'],
            'DATA_BREACH': ['CONFIDENTIALITY', 'INDEMNIFICATION'],
            'PAYMENT_DEFAULT': ['PAYMENT_TERMS', 'TERMINATION'],
        }

        # TODO: Add risk nodes and mitigation relationships
        # YOUR CODE HERE:
        for risk, mitigating_clauses in risk_mitigations.items():
            # Add risk node with type='RISK', color='#9b59b6'
            # Add MITIGATES edges from clauses to risk
            pass

print("ContractKnowledgeGraph class started")
print("TODO: Implement _build_contract_ontology()")

In [ ]:
# ============================================================================
# WEEK 6.2: KNOWLEDGE GRAPH METHODS
# ============================================================================

def add_contract(self, doc: Dict):
    """
    Add a contract document to the knowledge graph.

    Args:
        doc: Document dictionary with doc_id, filename, category

    TODO: Implement this method
    """

    doc_id = doc.get('doc_id', 'UNKNOWN')

    # TODO: Add contract node
    # Hint: self.graph.add_node(doc_id, type='CONTRACT', color='#2ecc71', ...)

    # YOUR CODE HERE:


    # TODO: Link to category
    category = doc.get('category', 'unknown')
    # YOUR CODE HERE:


def get_risk_mitigations(self, risk: str) -> List[str]:
    """
    Get clauses that mitigate a specific risk.

    Args:
        risk: Risk type name

    Returns:
        List of mitigating clause names

    TODO: Implement this method

    Hints:
    - Use self.graph.in_edges(risk, data=True)
    - Filter by relation='MITIGATES'
    """

    # YOUR CODE HERE:
    mitigations = []


    return mitigations

def get_clause_concerns(self, clause: str) -> List[str]:
    """
    Get concerns addressed by a clause type.

    Args:
        clause: Clause type name

    Returns:
        List of concern names

    TODO: Implement this method
    """

    # YOUR CODE HERE:
    concerns = []


    return concerns

# Add methods to class
ContractKnowledgeGraph.add_contract = add_contract
ContractKnowledgeGraph.get_risk_mitigations = get_risk_mitigations
ContractKnowledgeGraph.get_clause_concerns = get_clause_concerns

print("Query methods added to ContractKnowledgeGraph")

In [ ]:
# ============================================================================
# WEEK 6.3: VISUALIZATION METHODS
# ============================================================================

def visualize_matplotlib(self):
    """
    Create matplotlib visualization of the knowledge graph.

    TODO: Implement this method

    Hints:
    - Use nx.spring_layout for positioning
    - Use nx.draw_networkx_nodes, nx.draw_networkx_labels, nx.draw_networkx_edges
    - Color nodes based on their 'color' attribute
    """

    plt.figure(figsize=(16, 12))

    # TODO: Get colors from node attributes
    # Hint: colors = [self.graph.nodes[n].get('color', '#95a5a6') for n in self.graph.nodes()]

    # YOUR CODE HERE:
    colors = []


    # TODO: Calculate layout
    # Hint: pos = nx.spring_layout(self.graph, k=2, iterations=50, seed=42)

    # YOUR CODE HERE:
    pos = None


    # TODO: Draw nodes, labels, and edges
    # YOUR CODE HERE:


    plt.title("Contract Knowledge Graph", fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('contract_knowledge_graph.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("Visualization saved to contract_knowledge_graph.png")

def visualize_interactive(self, filename: str = "contract_kg.html"):
    """
    Create interactive PyVis visualization.

    TODO: Implement this method

    Hints:
    - Create Network(height="600px", width="100%", directed=True, notebook=True)
    - Add nodes with net.add_node(id, label=..., color=..., title=...)
    - Add edges with net.add_edge(source, target, title=...)
    """

    # TODO: Create PyVis network
    # YOUR CODE HERE:
    net = Network(height="600px", width="100%", directed=True, notebook=True)


    # TODO: Add nodes from graph
    # YOUR CODE HERE:


    # TODO: Add edges from graph
    # YOUR CODE HERE:


    # Save graph
    net.save_graph(filename)
    print(f"Interactive graph saved to {filename}")
    return filename

# Add methods to class
ContractKnowledgeGraph.visualize_matplotlib = visualize_matplotlib
ContractKnowledgeGraph.visualize_interactive = visualize_interactive

print("Visualization methods added")

In [ ]:
# ============================================================================
# INITIALIZE AND POPULATE KNOWLEDGE GRAPH
# ============================================================================

# Create knowledge graph
contract_kg = ContractKnowledgeGraph()

# Add all processed contracts
for doc in all_docs_flat:
    contract_kg.add_contract(doc)

# Log to Langfuse
langfuse.trace(
    name="knowledge-graph-built",
    session_id=SESSION_ID,
    input={"contracts_added": len(all_docs_flat)},
    output={
        "nodes": contract_kg.graph.number_of_nodes(),
        "edges": contract_kg.graph.number_of_edges()
    }
)

print(f"Contract Knowledge Graph Built:")
print(f"  Nodes: {contract_kg.graph.number_of_nodes()}")
print(f"  Edges: {contract_kg.graph.number_of_edges()}")

In [ ]:
# ============================================================================
# VISUALIZE KNOWLEDGE GRAPH
# ============================================================================

print("KNOWLEDGE GRAPH VISUALIZATION")
print("=" * 60)

# Create visualizations
try:
    contract_kg.visualize_matplotlib()
except Exception as e:
    print(f"Matplotlib visualization error: {e}")

try:
    contract_kg.visualize_interactive("contract_knowledge_graph.html")
except Exception as e:
    print(f"PyVis visualization error: {e}")

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH QUERIES
# ============================================================================

print("\nKNOWLEDGE GRAPH QUERIES")
print("=" * 60)

# Query 1: Risk mitigations
print("\n1. Risk Mitigation Strategies:")
for risk in ['UNLIMITED_LIABILITY', 'IP_INFRINGEMENT', 'DATA_BREACH']:
    mitigations = contract_kg.get_risk_mitigations(risk)
    print(f"   {risk}: {mitigations}")

# Query 2: Clause concerns
print("\n2. Clause Concerns:")
for clause in ['INDEMNIFICATION', 'CONFIDENTIALITY', 'TERMINATION']:
    concerns = contract_kg.get_clause_concerns(clause)
    print(f"   {clause}: {concerns}")

# Query 3: Graph statistics
print("\n3. Graph Statistics:")
print(f"   Total nodes: {contract_kg.graph.number_of_nodes()}")
print(f"   Total edges: {contract_kg.graph.number_of_edges()}")

# Node type distribution
type_counts = {}
for _, data in contract_kg.graph.nodes(data=True):
    t = data.get('type', 'OTHER')
    type_counts[t] = type_counts.get(t, 0) + 1

print("\n4. Node Type Distribution:")
for t, count in sorted(type_counts.items()):
    print(f"   {t}: {count}")

langfuse.flush()

## Checkpoint: Week 6 Verification

In [ ]:
# ============================================================================
# WEEK 6 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 6 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Knowledge graph initialized
checks.append(("Knowledge graph initialized", contract_kg is not None))

# Check 2: Graph has nodes
try:
    node_count = contract_kg.graph.number_of_nodes()
    checks.append(("Graph has nodes", node_count > 0))
except:
    checks.append(("Graph has nodes", False))

# Check 3: Graph has edges
try:
    edge_count = contract_kg.graph.number_of_edges()
    checks.append(("Graph has edges", edge_count > 0))
except:
    checks.append(("Graph has edges", False))

# Check 4: Query methods work
try:
    mitigations = contract_kg.get_risk_mitigations("UNLIMITED_LIABILITY")
    checks.append(("get_risk_mitigations works", isinstance(mitigations, list)))
except Exception as e:
    checks.append(("get_risk_mitigations works", False))

# Check 5: Contracts added
try:
    contract_nodes = [n for n, d in contract_kg.graph.nodes(data=True) if d.get('type') == 'CONTRACT']
    checks.append(("Contracts added to graph", len(contract_nodes) >= 0))  # May be 0 if no docs
except:
    checks.append(("Contracts added to graph", False))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 6 CHECKPOINTS PASSED! Ready for Week 7.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")

---

# WEEK 7: Advanced Observability & Metrics

**Difficulty: Medium** - Add custom metrics and monitoring.

---

## Learning Objectives

By the end of Week 7, you will:
- [ ] Review traces in Langfuse
- [ ] Add custom scores to traces
- [ ] Implement quality metrics
- [ ] Create observability summaries

---

## Key Concept: Production Observability

In production, you need to:
1. **Monitor**: Track system health and performance
2. **Evaluate**: Score output quality over time
3. **Debug**: Trace issues to specific calls
4. **Optimize**: Identify bottlenecks and costs

In [ ]:
# ============================================================================
# WEEK 7.1: OBSERVABILITY SUMMARY
# ============================================================================

print("LANGFUSE OBSERVABILITY SUMMARY")
print("=" * 70)
print(f"\nSession ID: {SESSION_ID}")
print(f"\nTraces created during this session:")
print()
print("Week 1: Environment Setup")
print("  - taxonomy-definition")
print("  - data-discovery")
print("  - test-embedding-*, test-completion-*")
print()
print("Week 2: Document Processing & EDA")
print("  - process-{category}")
print("  - entity-extraction-batch")
print()
print("Week 3: Vector Store & Embeddings")
print("  - vectorstore-init")
print("  - index-contracts_all")
print("  - embed-{doc_id}")
print("  - semantic-search")
print()
print("Week 4: Single Agent Design")
print("  - legal-agent-{doc_id}")
print("  - financial-agent-{doc_id}")
print("  - operational-agent-{doc_id}")
print()
print("Week 5: Multi-Agent Orchestration")
print("  - orchestration-{doc_id}")
print()
print("Week 6: Knowledge Graph")
print("  - knowledge-graph-init")
print("  - knowledge-graph-built")
print()
print(f"View all traces at: {os.environ.get('LANGFUSE_HOST')}/project")

## 7.2 Adding Custom Scores

**TODO:** Add custom quality scores to your traces.

### Hints:
- Use `langfuse.score(trace_id=..., name=..., value=..., comment=...)`
- Scores can be numeric (0-1) or categorical
- Useful for tracking quality over time

In [ ]:
# ============================================================================
# WEEK 7.2: ADD CUSTOM SCORES
# ============================================================================

print("\nADDING CUSTOM SCORES TO TRACES")
print("=" * 60)

# TODO: Score the orchestration result
# Hint: Convert risk level to numeric score

if 'orch_result' in dir() and orch_result:
    doc_id = orch_result['doc_id']
    overall_risk = orch_result['overall_risk']

    # TODO: Map risk to score (higher = better/lower risk)
    risk_score_map = {
        'LOW': 1.0,
        'MEDIUM': 0.7,
        'HIGH': 0.4,
        'CRITICAL': 0.1,
        'UNKNOWN': 0.5
    }

    # YOUR CODE HERE:
    score = risk_score_map.get(overall_risk, 0.5)


    # TODO: Create trace and add score
    # Hint:
    # score_trace = langfuse.trace(name=..., session_id=SESSION_ID, ...)
    # langfuse.score(trace_id=score_trace.id, name="contract-risk-score", value=score, ...)

    # YOUR CODE HERE:


    print(f"\nScored {doc_id}:")
    print(f"  Risk Level: {overall_risk}")
    print(f"  Score: {score}")
else:
    print("No orchestration results to score.")

langfuse.flush()
print("\nScores sent to Langfuse!")

## 7.3 Custom Metrics Implementation

**TODO:** Implement a metrics collector class.

### Hints:
- Track latency, token usage, success rates
- Calculate aggregates over time
- Log metrics to Langfuse

In [ ]:
# ============================================================================
# WEEK 7.3: METRICS COLLECTOR
# ============================================================================

from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class MetricsCollector:
    """
    Collect and aggregate system metrics.

    TODO: Implement these methods:
    - record_latency(): Track operation latencies
    - record_success(): Track success/failure rates
    - get_summary(): Return aggregated metrics
    """

    latencies: Dict[str, List[float]] = field(default_factory=dict)
    successes: Dict[str, int] = field(default_factory=dict)
    failures: Dict[str, int] = field(default_factory=dict)
    start_time: datetime = field(default_factory=datetime.now)

    def record_latency(self, operation: str, latency_ms: float):
        """
        Record latency for an operation.

        TODO: Implement this method
        """
        # YOUR CODE HERE:
        if operation not in self.latencies:
            self.latencies[operation] = []
        self.latencies[operation].append(latency_ms)

    def record_success(self, operation: str, success: bool):
        """
        Record success or failure for an operation.

        TODO: Implement this method
        """
        # YOUR CODE HERE:
        if success:
            self.successes[operation] = self.successes.get(operation, 0) + 1
        else:
            self.failures[operation] = self.failures.get(operation, 0) + 1

    def get_summary(self) -> Dict:
        """
        Get aggregated metrics summary.

        TODO: Implement this method

        Returns dict with:
        - avg_latency per operation
        - success_rate per operation
        - total_operations
        """

        # YOUR CODE HERE:
        summary = {
            'session_duration': (datetime.now() - self.start_time).total_seconds(),
            'latencies': {},
            'success_rates': {}
        }

        # TODO: Calculate average latencies
        for op, times in self.latencies.items():
            summary['latencies'][op] = {
                'avg_ms': sum(times) / len(times) if times else 0,
                'count': len(times)
            }

        # TODO: Calculate success rates
        for op in set(list(self.successes.keys()) + list(self.failures.keys())):
            s = self.successes.get(op, 0)
            f = self.failures.get(op, 0)
            total = s + f
            summary['success_rates'][op] = {
                'rate': s / total if total > 0 else 0,
                'total': total
            }

        return summary

# Initialize metrics collector
metrics = MetricsCollector()

print("MetricsCollector initialized")

In [ ]:
# ============================================================================
# WEEK 7.4: SIMULATED METRICS
# ============================================================================

import random

# Simulate some metrics (in production, these would come from actual operations)
print("\nSIMULATED METRICS COLLECTION")
print("=" * 60)

operations = ['embedding', 'legal_agent', 'financial_agent', 'operational_agent', 'search']

for _ in range(20):
    op = random.choice(operations)
    latency = random.uniform(100, 2000)  # 100ms to 2s
    success = random.random() > 0.1  # 90% success rate

    metrics.record_latency(op, latency)
    metrics.record_success(op, success)

# Get summary
summary = metrics.get_summary()

print(f"\nSession Duration: {summary['session_duration']:.1f} seconds")
print("\nLatencies (avg ms):")
for op, data in summary['latencies'].items():
    print(f"  {op}: {data['avg_ms']:.1f}ms ({data['count']} calls)")

print("\nSuccess Rates:")
for op, data in summary['success_rates'].items():
    print(f"  {op}: {data['rate']*100:.1f}% ({data['total']} total)")

# Log metrics to Langfuse
langfuse.trace(
    name="metrics-summary",
    session_id=SESSION_ID,
    input={"type": "metrics"},
    output=summary
)

langfuse.flush()

## Checkpoint: Week 7 Verification

In [ ]:
# ============================================================================
# WEEK 7 CHECKPOINT - VALIDATION
# ============================================================================

print("WEEK 7 CHECKPOINT - Verification")
print("=" * 60)

checks = []

# Check 1: Metrics collector exists
checks.append(("MetricsCollector initialized", metrics is not None))

# Check 2: Latencies recorded
try:
    has_latencies = len(metrics.latencies) > 0
    checks.append(("Latencies recorded", has_latencies))
except:
    checks.append(("Latencies recorded", False))

# Check 3: Success rates recorded
try:
    has_successes = len(metrics.successes) > 0 or len(metrics.failures) > 0
    checks.append(("Success rates recorded", has_successes))
except:
    checks.append(("Success rates recorded", False))

# Check 4: Summary works
try:
    summary = metrics.get_summary()
    checks.append(("get_summary() works", 'latencies' in summary and 'success_rates' in summary))
except:
    checks.append(("get_summary() works", False))

# Check 5: Session ID exists
checks.append(("Session tracking active", SESSION_ID is not None))

# Display results
all_passed = True
for check_name, check_result in checks:
    status = "PASS" if check_result else "FAIL"
    print(f"  [{status}] {check_name}")
    if not check_result:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("ALL WEEK 7 CHECKPOINTS PASSED! Ready for Week 8.")
else:
    print("Some checkpoints FAILED. Review your implementation above.")